In [1]:
import os
import sys
import pandas as pd
import yaml
import numpy as np

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from utils.Sampler import Sampler, BuildstockBatchArguments, MapHPXML
from builder.EUEMr import Attribut_EUEMr

#from utils.Master_genereBN import Master_genereBN
#from builder.EUEMr import FormatageEUEMr, EUEMr

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [2]:
#Données BN
with open("../dataStructure/Bn.yml", 'r') as file:
        NOEUD_EUEMr, LIST_Dict, dict_info = yaml.safe_load(file)

#données csv
Bba = BuildstockBatchArguments()
dctcsv_sampler =Bba.dct_housing_characteristics

In [3]:
#Traitement des données csv
for k in dctcsv_sampler.keys():
    dctcsv_sampler[k]["Nom"] = k
    dctcsv_sampler[k]["Échantillonneur"] = "BuildstockBatchArguments"
    dctcsv_sampler[k]["Valeurs"] = list(dctcsv_sampler[k]["Option"].values())
    dctcsv_sampler[k]["Dépendance (parents)"] = list(dctcsv_sampler[k]["Dependency"].values())

for k in dctcsv_sampler.keys():
    dctcsv_sampler[k]["Dépendance (enfants)"] = []
    for kk in dctcsv_sampler.keys():
        if k in dctcsv_sampler[kk]["Dépendance (parents)"]:
            dctcsv_sampler[k]["Dépendance (enfants)"].append(kk)


#Ajout des variables enfant dans le bn provenant des csv

for k in dict_info.keys():
    for kkbn in dctcsv_sampler.keys():
        if k in dctcsv_sampler[kkbn]["Dépendance (parents)"]:
            dict_info[k]["Dépendance (enfants)"].append(kkbn)

pdcsv_sampler = pd.DataFrame(dctcsv_sampler).T
pdcsv_sampler = pdcsv_sampler[["Nom", "Description", "Valeurs", "Dépendance (parents)", "Dépendance (enfants)", "Échantillonneur", "Source"]]#,


pdDescription = pd.DataFrame.from_dict(dict_info).T.set_index("id").sort_index().reset_index()[["id", "Nom", "Description", "Valeurs", "Dépendance (parents)","Dépendance (enfants)", "Échantillonneur", "Source"]]
#pdDescription["Échantillonneur"] = "Réseau Bayesien"
#pdDescription["Source"] = "EUEMR"



Data_description = pd.concat([pdDescription, pdcsv_sampler], axis=0).reset_index(drop=True)

Data_description.to_csv("../dataStructure/Data_description.csv", index=False)

In [4]:
Data_description

,id,Nom,Description,Valeurs,Dépendance (parents),Dépendance (enfants),Échantillonneur,Source
0,0.0,Territoire_HQ,Territoire HQ,"[Est et Nord du Québec, Laurentides, Montmoren...",[],[Region_Administrative],Réseau Bayesien,EUEMR modifié
1,1.0,Region_Administrative,Zone du Québec,"[Bas-Saint-Laurent, Capitale-Nationale, Centre...",[Territoire_HQ],"[Type_Logement, Nombre_Etages, Spa_Presence, P...",Réseau Bayesien,EUEMR modifié
2,2.0,Type_Logement,De quel genre d'habitation s'agit-il?,"[Collective, Triplex, Duplex, Maison en rangee...",[Region_Administrative],"[Type_Batiment, Nombre_Etages, Nombre_Pieces, ...",Réseau Bayesien,EUEMR modifié
3,3.0,Type_Batiment,De quel genre d'habitation s'agit-il?,"[Collective, Plex, Maison]",[Type_Logement],[],Réseau Bayesien,EUEMR modifié
4,4.0,Nombre_Etages,Nombre d'étages habitables dans la résidence (...,"[Un étage, Deux étages, Trois étages et plus]","[Region_Administrative, Type_Logement]","[Nombre_Pieces, Presence_SousSol, Geometry Sto...",Réseau Bayesien,EUEMR modifié
5,5.0,Nombre_Pieces,Nombre de pièces dans la résidence (incluant l...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[Type_Logement, Nombre_Etages]","[Nombre_Personnes, Superficie_Totale]",Réseau Bayesien,EUEMR modifié
6,6.0,Superficie_Totale,Quelle est la superficie TOTALE habitable de v...,"[[1 - 500), [500 - 1000), [1000 - 1500), [1500...",[Nombre_Pieces],[],Réseau Bayesien,EUEMR modifié
7,7.0,Presence_SousSol,Votre résidence comporte-elle un sous-sol ou v...,"[Sous sol 6 pied, Vide sanitaire moins 6 pieds...","[Type_Logement, Nombre_Etages]",[],Réseau Bayesien,EUEMR modifié
8,8.0,Nombre_Personnes,"En vous incluant, combien de personnes habiten...","[1, 2, 3, 4, 5 et plus]",[Nombre_Pieces],[],Réseau Bayesien,EUEMR modifié
9,9.0,Presence_Garage,Avez-vous un garage?,"[Pas de Garage, Garage non chauffé, Garage cha...",[Type_Logement],[],Réseau Bayesien,EUEMR modifié


In [5]:
#EUEMr_description
EUEMr_description = {}
found_dicts = []
for attr_name, attr_value in Attribut_EUEMr.__dict__.items():
    if isinstance(attr_value, dict):
        found_dicts.append({**{"Code": attr_name}, **attr_value})
found_dicts
EUEMr_description = pd.DataFrame(found_dicts)[["Code", "Description", "Label", "IdLabel", "Type"]]#
EUEMr_description.to_csv("../dataStructure/EUEMr_description.csv", index=False)
EUEMr_description

,Code,Description,Label,IdLabel,Type
0,QA1,Quel est votre lien avec ce logement ? En êtes...,"[Le propriétaire / Co-propriétaire, Le locatai...","[0, 1]",discrete
1,QA4,De quel genre d'habitation s'agit-il?,"[Immeuble de 4 à 8 appartements, Immeuble de 9...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]",discrete
2,QC1R,Principale source d'énergie utilisée pour le c...,"[Électricité, Gaz naturel, Mazout, Bois seul o...","[0, 1, 2, 3, 4, 5]",discrete
3,CONS_AN,Consommation annuelle d'électricité en kWh,None,None,Double
4,QA6M,Année de construction de l'habitation,None,None,Entier
...,...,...,...,...,...
113,QT2R,Combien avez-vous de voiture(s) 100 % électriq...,"[Aucune, Une, Deux, Trois]","[0, 1, 2, 3]",discrete
114,QT3,Combien avez-vous de voiture(s) électrique(s) ...,"[Aucune, Une, Deux]","[0, 1, 2]",discrete
115,QT3R,Combien avez-vous de voiture(s) électrique(s) ...,"[Aucune, Une, Deux]","[0, 1, 2]",discrete
116,QT2T3,Voiture(s) électrique? (Base totale),"[Ne possède aucune voiture électrique, Possède...","[0, 1, 2, 3]",discrete
